# Análise de dados TCP-CII

In [15]:
import pandas as pd

In [16]:
df = pd.read_csv('./T CELL/DENV 1 - T Cell Prediction - Class II.csv')
df

,seq #,peptide,start,end,peptide length,allele,peptide index,median binding percentile,netmhciipan_el-4.3 core,netmhciipan_el-4.3 score,netmhciipan_el-4.3 percentile
0,1,KNETWKLARASFIE,206,219,14,HLA-DRB1*09:01,110,0.07,WKLARASFI,0.911329,0.07
1,1,QYKFQADSPKRLSA,31,44,14,HLA-DRB3*01:01,75,0.08,FQADSPKRL,0.920957,0.08
2,1,KNETWKLARASFIEVK,206,221,16,HLA-DRB1*09:01,246,0.08,WKLARASFI,0.910584,0.08
3,1,QYKFQADSPKRLS,31,43,13,HLA-DRB3*01:01,7,0.09,FQADSPKRL,0.915925,0.09
4,1,QYKFQADSPKRLSAA,31,45,15,HLA-DRB3*01:01,143,0.10,FQADSPKRL,0.912478,0.10
...,...,...,...,...,...,...,...,...,...,...,...
16411,1,WCCRSCTLPPLRFKGEDGCW,311,330,20,HLA-DRB1*04:01,537,100.00,CRSCTLPPL,0.000000,100.00
16412,1,WCCRSCTLPPLRFKGEDGCW,311,330,20,HLA-DRB1*04:05,537,100.00,PLRFKGEDG,0.000000,100.00
16413,1,WCCRSCTLPPLRFKGEDGCW,311,330,20,HLA-DRB3*01:01,537,100.00,LRFKGEDGC,0.000000,100.00
16414,1,WCCRSCTLPPLRFKGEDGCWY,311,331,21,HLA-DRB1*04:01,604,100.00,FKGEDGCWY,0.000000,100.00


## Selecionando Epítopos com median binding percentile menor que 5.

In [17]:
df_mbp_m5 = df[df['median binding percentile'] < 5].copy()
df_mbp_m5

,seq #,peptide,start,end,peptide length,allele,peptide index,median binding percentile,netmhciipan_el-4.3 core,netmhciipan_el-4.3 score,netmhciipan_el-4.3 percentile
0,1,KNETWKLARASFIE,206,219,14,HLA-DRB1*09:01,110,0.07,WKLARASFI,0.911329,0.07
1,1,QYKFQADSPKRLSA,31,44,14,HLA-DRB3*01:01,75,0.08,FQADSPKRL,0.920957,0.08
2,1,KNETWKLARASFIEVK,206,221,16,HLA-DRB1*09:01,246,0.08,WKLARASFI,0.910584,0.08
3,1,QYKFQADSPKRLS,31,43,13,HLA-DRB3*01:01,7,0.09,FQADSPKRL,0.915925,0.09
4,1,QYKFQADSPKRLSAA,31,45,15,HLA-DRB3*01:01,143,0.10,FQADSPKRL,0.912478,0.10
...,...,...,...,...,...,...,...,...,...,...,...
694,1,VTNEVHTWTEQYKFQADSPK,21,40,20,HLA-DPA1*02:01/DPB1*05:01,479,4.90,KYQETWTHV,0.051623,4.90
695,1,KNETWKLARASFIE,206,219,14,HLA-DRB3*01:01,110,4.90,WKLARASFI,0.050958,4.90
696,1,DFDLCEGTTVVVDEHCGNR,276,294,19,HLA-DQA1*03:01/DQB1*03:02,463,4.90,EGTTVVVDE,0.036866,4.90
697,1,AKIIGADVQNTTFIIDGPNT,121,140,20,HLA-DPA1*01:03/DPB1*02:01,499,4.90,VQNTTFIID,0.025151,4.90


## Agrupando por pepitideos e agregando colunas pertinentes.

In [18]:
epitopos_repetidos = (
    df_mbp_m5
    .groupby('peptide', as_index=False)
    .agg(
        start=("start", "first"),
        end=("end", "first"),
        qte_de_alelos=("allele", "nunique"),
        median_binding_percentile=(
            "median binding percentile",
            "median"
        ),
        alelos=(
            "allele",
            lambda x: ", ".join(sorted(x.unique()))
        )
    ))

epitopos_repetidos

,peptide,start,end,qte_de_alelos,median_binding_percentile,alelos
0,AAIKDSKAVHADM,186,198,6,3.30,"HLA-DQA1*01:02/DQB1*06:02, HLA-DQA1*05:01/DQB1..."
1,AAIKDSKAVHADMG,186,199,4,3.25,"HLA-DQA1*01:02/DQB1*06:02, HLA-DQA1*05:01/DQB1..."
2,AAIKDSKAVHADMGY,186,200,2,3.15,"HLA-DQA1*01:02/DQB1*06:02, HLA-DQA1*05:01/DQB1..."
3,AAIKDSKAVHADMGYW,186,201,1,3.40,HLA-DQA1*05:01/DQB1*03:01
4,AAIKDSKAVHADMGYWI,186,202,1,4.70,HLA-DQA1*05:01/DQB1*03:01
...,...,...,...,...,...,...
306,YGMEIRPVKEKEENL,331,345,1,4.00,HLA-DQA1*04:01/DQB1*04:02
307,YGMEIRPVKEKEENLV,331,346,1,4.40,HLA-DQA1*04:01/DQB1*04:02
308,YRPGYFTQTAGPW,256,268,1,4.40,HLA-DQA1*05:01/DQB1*02:01
309,YRPGYFTQTAGPWH,256,269,1,3.40,HLA-DQA1*05:01/DQB1*02:01


## Filtragem por qte_de_alelos

In [19]:
filtarar_por_qte_de_alelos = 2

In [20]:
# Filtro do número de alelos
epitopos_repetidos = epitopos_repetidos[
    epitopos_repetidos["qte_de_alelos"] >= filtarar_por_qte_de_alelos
].reset_index(drop=True)

epitopos_repetidos

,peptide,start,end,qte_de_alelos,median_binding_percentile,alelos
0,AAIKDSKAVHADM,186,198,6,3.300,"HLA-DQA1*01:02/DQB1*06:02, HLA-DQA1*05:01/DQB1..."
1,AAIKDSKAVHADMG,186,199,4,3.250,"HLA-DQA1*01:02/DQB1*06:02, HLA-DQA1*05:01/DQB1..."
2,AAIKDSKAVHADMGY,186,200,2,3.150,"HLA-DQA1*01:02/DQB1*06:02, HLA-DQA1*05:01/DQB1..."
3,ADMGYWIESEKNE,196,208,2,2.250,"HLA-DPA1*02:01/DPB1*05:01, HLA-DQA1*01:01/DQB1..."
4,ADMGYWIESEKNET,196,209,2,2.485,"HLA-DPA1*02:01/DPB1*05:01, HLA-DQA1*01:01/DQB1..."
...,...,...,...,...,...,...
176,VTNEVHTWTEQYKFQADSPK,21,40,3,3.400,"HLA-DPA1*02:01/DPB1*05:01, HLA-DQA1*04:01/DQB1..."
177,VTNEVHTWTEQYKFQADSPKR,21,41,3,3.600,"HLA-DQA1*04:01/DQB1*04:02, HLA-DRB1*11:01, HLA..."
178,WIESEKNETWKLARASFIE,201,219,3,1.700,"HLA-DQA1*01:01/DQB1*05:01, HLA-DRB1*07:01, HLA..."
179,WIESEKNETWKLARASFIEV,201,220,3,1.400,"HLA-DRB1*01:01, HLA-DRB1*07:01, HLA-DRB1*09:01"


In [21]:
epitopos_repetidos = (
    epitopos_repetidos
    .sort_values(
        ["median_binding_percentile", "qte_de_alelos"],
        ascending=[True, False]
    )
    .reset_index(drop=True)
)

epitopos_repetidos

,peptide,start,end,qte_de_alelos,median_binding_percentile,alelos
0,KNETWKLARASFI,206,218,3,0.430,"HLA-DRB1*01:01, HLA-DRB1*07:01, HLA-DRB1*09:01"
1,GIFTTNIWLKLRD,161,173,4,0.585,"HLA-DPA1*01:03/DPB1*02:01, HLA-DPA1*01:03/DPB1..."
2,PQPMEHKYSWKSWGKAKIIGA,106,126,2,0.740,"HLA-DPA1*01:03/DPB1*02:01, HLA-DRB1*13:02"
3,HKYSWKSWGKAKIIG,111,125,3,0.760,"HLA-DPA1*01:03/DPB1*02:01, HLA-DRB1*11:01, HLA..."
4,HTWTEQYKFQADSPKRLSA,26,44,3,0.770,"HLA-DRB3*01:01, HLA-DRB3*02:02, HLA-DRB5*01:01"
...,...,...,...,...,...,...
176,GSGIFVTNEVHTWTEQYKFQ,16,35,2,4.350,"HLA-DQA1*04:01/DQB1*04:02, HLA-DRB3*02:02"
177,ISQHNYRPGYFTQTA,251,265,2,4.400,"HLA-DPA1*01:03/DPB1*02:01, HLA-DPA1*01:03/DPB1..."
178,ISNELNHILLENDMKF,71,86,2,4.550,"HLA-DPA1*02:01/DPB1*01:01, HLA-DPA1*03:01/DPB1..."
179,NHILLENDMKFTV,76,88,3,4.700,"HLA-DPA1*02:01/DPB1*05:01, HLA-DRB1*03:01, HLA..."


## Separando epítopos e criando arquivo FASTA para IEDB analysis resource

In [22]:
pepitides = epitopos_repetidos.peptide

with open("./peptideos.fasta", "w") as f:
    for i, peptide in enumerate(pepitides, start=1):
        f.write(f">NP {i}\n")
        f.write(f"{peptide}\n")
        
pepitides

0              KNETWKLARASFI
1              GIFTTNIWLKLRD
2      PQPMEHKYSWKSWGKAKIIGA
3            HKYSWKSWGKAKIIG
4        HTWTEQYKFQADSPKRLSA
               ...          
176     GSGIFVTNEVHTWTEQYKFQ
177          ISQHNYRPGYFTQTA
178         ISNELNHILLENDMKF
179            NHILLENDMKFTV
180      SKAVHADMGYWIESEKNET
Name: peptide, Length: 181, dtype: str

### Seqkit remove sequências proteicas contendo gaps e *.

In [23]:
!seqkit grep -s -v -r -p '[-*]' './Fastas/denv1_NS1_proteinas.fa' > DENV1_seq_filter_all.fasta

/bin/bash: line 1: seqkit: command not found


### Resultado IEDB analysis resource

In [24]:
conservacy_result = pd.read_csv('./ConservancyResult_tcell_2.csv')
conservacy_result

,Epitope #,Epitope name,Epitope sequence,Epitope length,Percent of protein sequence matches at identity <= 100%,Minimum identity,Maximum identity,View details
0,1,NP 1,KNETWKLARASFI,13,89.46% (908/1015),61.54%,100.00%,NaN
1,2,NP 2,GIFTTNIWLKLRD,13,88.97% (903/1015),84.62%,100.00%,NaN
2,3,NP 3,PQPMEHKYSWKSWGKAKIIGA,21,55.67% (565/1015),80.95%,100.00%,NaN
3,4,NP 4,HKYSWKSWGKAKIIG,15,55.67% (565/1015),73.33%,100.00%,NaN
4,5,NP 5,HTWTEQYKFQADSPKRLSA,19,98.62% (1001/1015),89.47%,100.00%,NaN
...,...,...,...,...,...,...,...,...
176,177,NP 177,GSGIFVTNEVHTWTEQYKFQ,20,99.31% (1008/1015),95.00%,100.00%,NaN
177,178,NP 178,ISQHNYRPGYFTQTA,15,93.10% (945/1015),93.33%,100.00%,NaN
178,179,NP 179,ISNELNHILLENDMKF,16,85.32% (866/1015),62.50%,100.00%,NaN
179,180,NP 180,NHILLENDMKFTV,13,86.21% (875/1015),61.54%,100.00%,NaN


### Merge da coluna qte_de_alelos ao dataframe conservacy_result

In [25]:
# qte_de_alelos
conservacy_result = conservacy_result.merge(
    epitopos_repetidos[["peptide", "qte_de_alelos"]],
    left_on="Epitope sequence",
    right_on="peptide",
    how="left"
).drop(columns=("peptide")).drop(columns=("View details"))

# alelos
conservacy_result = conservacy_result.merge(
    epitopos_repetidos[["peptide", "alelos"]],
    left_on="Epitope sequence",
    right_on="peptide",
    how="left"
).drop(columns=("peptide")).drop(columns=("Epitope #"))

conservacy_result

,Epitope name,Epitope sequence,Epitope length,Percent of protein sequence matches at identity <= 100%,Minimum identity,Maximum identity,qte_de_alelos,alelos
0,NP 1,KNETWKLARASFI,13,89.46% (908/1015),61.54%,100.00%,3,"HLA-DRB1*01:01, HLA-DRB1*07:01, HLA-DRB1*09:01"
1,NP 2,GIFTTNIWLKLRD,13,88.97% (903/1015),84.62%,100.00%,4,"HLA-DPA1*01:03/DPB1*02:01, HLA-DPA1*01:03/DPB1..."
2,NP 3,PQPMEHKYSWKSWGKAKIIGA,21,55.67% (565/1015),80.95%,100.00%,2,"HLA-DPA1*01:03/DPB1*02:01, HLA-DRB1*13:02"
3,NP 4,HKYSWKSWGKAKIIG,15,55.67% (565/1015),73.33%,100.00%,3,"HLA-DPA1*01:03/DPB1*02:01, HLA-DRB1*11:01, HLA..."
4,NP 5,HTWTEQYKFQADSPKRLSA,19,98.62% (1001/1015),89.47%,100.00%,3,"HLA-DRB3*01:01, HLA-DRB3*02:02, HLA-DRB5*01:01"
...,...,...,...,...,...,...,...,...
176,NP 177,GSGIFVTNEVHTWTEQYKFQ,20,99.31% (1008/1015),95.00%,100.00%,2,"HLA-DQA1*04:01/DQB1*04:02, HLA-DRB3*02:02"
177,NP 178,ISQHNYRPGYFTQTA,15,93.10% (945/1015),93.33%,100.00%,2,"HLA-DPA1*01:03/DPB1*02:01, HLA-DPA1*01:03/DPB1..."
178,NP 179,ISNELNHILLENDMKF,16,85.32% (866/1015),62.50%,100.00%,2,"HLA-DPA1*02:01/DPB1*01:01, HLA-DPA1*03:01/DPB1..."
179,NP 180,NHILLENDMKFTV,13,86.21% (875/1015),61.54%,100.00%,3,"HLA-DPA1*02:01/DPB1*05:01, HLA-DRB1*03:01, HLA..."


### Gerando a coluna percent_match para filtrar os epitopos com percentagem de match maior que 50%

In [26]:
col = "Percent of protein sequence matches at identity <= 100%"

conservacy_result["percent_match"] = (
    conservacy_result[col]
    .astype(str)
    .str.extract(r"(\d+(?:\.\d+)?)")[0]
    .astype(float)
)

conservacy_result

,Epitope name,Epitope sequence,Epitope length,Percent of protein sequence matches at identity <= 100%,Minimum identity,Maximum identity,qte_de_alelos,alelos,percent_match
0,NP 1,KNETWKLARASFI,13,89.46% (908/1015),61.54%,100.00%,3,"HLA-DRB1*01:01, HLA-DRB1*07:01, HLA-DRB1*09:01",89.46
1,NP 2,GIFTTNIWLKLRD,13,88.97% (903/1015),84.62%,100.00%,4,"HLA-DPA1*01:03/DPB1*02:01, HLA-DPA1*01:03/DPB1...",88.97
2,NP 3,PQPMEHKYSWKSWGKAKIIGA,21,55.67% (565/1015),80.95%,100.00%,2,"HLA-DPA1*01:03/DPB1*02:01, HLA-DRB1*13:02",55.67
3,NP 4,HKYSWKSWGKAKIIG,15,55.67% (565/1015),73.33%,100.00%,3,"HLA-DPA1*01:03/DPB1*02:01, HLA-DRB1*11:01, HLA...",55.67
4,NP 5,HTWTEQYKFQADSPKRLSA,19,98.62% (1001/1015),89.47%,100.00%,3,"HLA-DRB3*01:01, HLA-DRB3*02:02, HLA-DRB5*01:01",98.62
...,...,...,...,...,...,...,...,...,...
176,NP 177,GSGIFVTNEVHTWTEQYKFQ,20,99.31% (1008/1015),95.00%,100.00%,2,"HLA-DQA1*04:01/DQB1*04:02, HLA-DRB3*02:02",99.31
177,NP 178,ISQHNYRPGYFTQTA,15,93.10% (945/1015),93.33%,100.00%,2,"HLA-DPA1*01:03/DPB1*02:01, HLA-DPA1*01:03/DPB1...",93.10
178,NP 179,ISNELNHILLENDMKF,16,85.32% (866/1015),62.50%,100.00%,2,"HLA-DPA1*02:01/DPB1*01:01, HLA-DPA1*03:01/DPB1...",85.32
179,NP 180,NHILLENDMKFTV,13,86.21% (875/1015),61.54%,100.00%,3,"HLA-DPA1*02:01/DPB1*05:01, HLA-DRB1*03:01, HLA...",86.21


### Sort e filtragem por percent_match e presença em alelos

In [27]:
# Parametros
percent_match_minimo = 95.0

In [28]:
# Filtro do Percent match
conservacy_result_filtered = (
    conservacy_result[conservacy_result["percent_match"] >= percent_match_minimo]
    .sort_values(
            by="percent_match", 
            ascending=False
        )
    ).reset_index(drop=True)

conservacy_result_filtered

,Epitope name,Epitope sequence,Epitope length,Percent of protein sequence matches at identity <= 100%,Minimum identity,Maximum identity,qte_de_alelos,alelos,percent_match
0,NP 80,HTWTEQYKFQADS,13,99.80% (1013/1015),92.31%,100.00%,2,"HLA-DRB1*08:02, HLA-DRB1*11:01",99.80
1,NP 102,HTWTEQYKFQADSP,14,99.80% (1013/1015),92.86%,100.00%,2,"HLA-DRB1*08:02, HLA-DRB1*11:01",99.80
2,NP 84,HTWTEQYKFQADSPKR,16,99.70% (1012/1015),93.75%,100.00%,2,"HLA-DRB1*11:01, HLA-DRB5*01:01",99.70
3,NP 156,HTWTEQYKFQADSPK,15,99.70% (1012/1015),93.33%,100.00%,2,"HLA-DRB1*11:01, HLA-DRB5*01:01",99.70
4,NP 7,HTWTEQYKFQADSPKRL,17,99.70% (1012/1015),94.12%,100.00%,2,"HLA-DRB3*01:01, HLA-DRB5*01:01",99.70
5,NP 118,VTNEVHTWTEQYKFQAD,17,99.51% (1010/1015),94.12%,100.00%,2,"HLA-DPA1*02:01/DPB1*05:01, HLA-DQA1*04:01/DQB1...",99.51
6,NP 121,VTNEVHTWTEQYKFQADSP,19,99.51% (1010/1015),94.74%,100.00%,3,"HLA-DPA1*02:01/DPB1*05:01, HLA-DQA1*04:01/DQB1...",99.51
7,NP 152,VTNEVHTWTEQYKFQADS,18,99.51% (1010/1015),94.44%,100.00%,3,"HLA-DPA1*02:01/DPB1*05:01, HLA-DQA1*04:01/DQB1...",99.51
8,NP 143,VTNEVHTWTEQYKFQ,15,99.51% (1010/1015),93.33%,100.00%,2,"HLA-DQA1*01:01/DQB1*05:01, HLA-DQA1*04:01/DQB1...",99.51
9,NP 161,VTNEVHTWTEQYKFQA,16,99.51% (1010/1015),93.75%,100.00%,3,"HLA-DPA1*02:01/DPB1*05:01, HLA-DQA1*01:01/DQB1...",99.51
